# Tutorial 5-2: Optimization and Regularization
**Course:** CSEN 140: Machine Learning/Data Mining  
**Instructor:** Dr. David C. Anastasiu

--- 
## Objective
In the previous tutorial, we found the 'best' model using a direct formula. Today, we explore how computers find that solution iteratively. We will explore:
1. **Gradient Descent (GD):** Implementing the update rule to 'walk down' the cost surface.
2. **Learning Rates:** Analyzing how the step size $\alpha$ affects convergence.
3. **Stochastic vs. Batch GD:** Comparing different ways to process training samples.
4. **Regularization:** Implementing Ridge and Lasso to prevent overfitting by penalizing large weights.

## 1. Implementing Batch Gradient Descent
The goal of Gradient Descent is to minimize the **Cost Function** $J(\theta)$. We calculate the partial derivative (the gradient) and move the parameters $\theta$ in the opposite direction.

The update rule for all parameters simultaneously is:
$$\theta := \theta - \alpha \nabla_{\theta} J(\theta)$$

Let's implement this from scratch for a simple linear relationship.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Generate synthetic data
np.random.seed(42)
X = 2 * np.random.rand(100, 1)
y = 4 + 3 * X + np.random.randn(100, 1)
X_b = np.c_[np.ones((100, 1)), X] # add x0 = 1

def batch_gradient_descent(X, y, lr, iterations):
    m = len(X)
    theta = np.random.randn(2,1)
    cost_history = []
    
    for i in range(iterations):
        gradients = 2/m * X.T.dot(X.dot(theta) - y)
        theta = theta - lr * gradients
        cost = np.mean(np.square(X.dot(theta) - y))
        cost_history.append(cost)
    return theta, cost_history

theta_final, costs = batch_gradient_descent(X_b, y, lr=0.1, iterations=50)

plt.plot(costs)
plt.title("Cost Reduction Over Iterations")
plt.xlabel("Iteration")
plt.ylabel("Mean Squared Error")
plt.show()
print(f"Final Theta (Intercept, Slope): {theta_final.ravel()}")

## 2. Visualizing the Cost Surface
As we saw on Slide 12, the cost function for linear regression is a convex 'bowl'. Let's visualize how the cost changes as we vary $\theta_0$ and $\theta_1$.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

theta0_vals = np.linspace(0, 8, 100)
theta1_vals = np.linspace(0, 6, 100)
J_vals = np.zeros((len(theta0_vals), len(theta1_vals)))

for i in range(len(theta0_vals)):
    for j in range(len(theta1_vals)):
        t = np.array([[theta0_vals[i]], [theta1_vals[j]]])
        J_vals[i, j] = np.mean(np.square(X_b.dot(t) - y))

T0, T1 = np.meshgrid(theta0_vals, theta1_vals)

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(T0, T1, J_vals.T, cmap='viridis', alpha=0.8)
ax.set_xlabel('theta_0 (Intercept)')
ax.set_ylabel('theta_1 (Slope)')
ax.set_zlabel('Cost J')
plt.title("The Convex Cost Surface (Slide 12)")
plt.show()

## 3. Stochastic Gradient Descent (SGD)
**Batch GD** uses the entire dataset to calculate one update. **Stochastic GD** updates the parameters after looking at just one random sample. It is much faster but 'noisier'—it bounces around the minimum rather than settling perfectly.

In [ ]:
theta = np.random.randn(2,1)
m = len(X_b)
n_epochs = 50
t0, t1 = 5, 50 # learning schedule hyperparameters

def learning_schedule(t):
    return t0 / (t + t1)

for epoch in range(n_epochs):
    for i in range(m):
        random_index = np.random.randint(m)
        xi = X_b[random_index:random_index+1]
        yi = y[random_index:random_index+1]
        gradients = 2 * xi.T.dot(xi.dot(theta) - yi)
        lr = learning_schedule(epoch * m + i)
        theta = theta - lr * gradients

print(f"SGD Final Theta: {theta.ravel()}")

## 4. Regularization: Fighting Overfitting
If our model is too complex, it might 'overfit'—learning the noise in the data rather than the signal. We add a penalty to the cost function to keep coefficients small.

### 4.1 Ridge Regression (L2 Penalty)
Adds $\lambda \sum \theta_j^2$ to the cost. It shrinks all coefficients towards zero but rarely makes them exactly zero.

### 4.2 Lasso Regression (L1 Penalty)
Adds $\lambda \sum |\theta_j|$ to the cost. It performs feature selection by forcing less important coefficients to be exactly zero.

In [ ]:
from sklearn.linear_model import Ridge, Lasso

# Let's create a scenario with many features, most of which are noise
X_large = np.random.randn(100, 20)
y_large = 3 * X_large[:, 0] + 2 * X_large[:, 1] + np.random.randn(100)

ridge = Ridge(alpha=10) # alpha is the 'lambda' from slides
lasso = Lasso(alpha=0.1)

ridge.fit(X_large, y_large)
lasso.fit(X_large, y_large)

plt.figure(figsize=(10, 5))
plt.plot(ridge.coef_, 's', label="Ridge Coefficients")
plt.plot(lasso.coef_, '^', label="Lasso Coefficients")
plt.axhline(0, color='black', linestyle='--')
plt.title("Impact of Regularization on Model Weights")
plt.xlabel("Feature Index")
plt.ylabel("Weight Magnitude")
plt.legend()
plt.show()

print(f"Number of Lasso weights that are zero: {np.sum(lasso.coef_ == 0)}")

## Summary
Optimization is how we transform a mathematical theory into a working computer program. 
1. **Gradient Descent** allows us to find the bottom of the error bowl iteratively.
2. **Learning Schedules** are vital for SGD to ensure the model eventually settles rather than jumping around forever.
3. **Regularization** (Ridge and Lasso) provides a lever to control the complexity of our model, ensuring it generalizes well to new data.